In [ ]:
import os
import math
import copy
import random
import numpy as np
import pandas as pd
import cv2

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models


# =========================================
# CONFIG
# =========================================
CSV_PATH = "pf_ratio_labels.csv"   # change to your CSV file
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
RANDOM_SEED = 42
NUM_WORKERS = 0   # start with 0 in Windows/Jupyter, can increase later

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =========================================
# REPRODUCIBILITY
# =========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)


# =========================================
# DATASET
# =========================================
class DualELDataset(Dataset):
    """
    Dataset for paired EL images:
    - low bias TIFF
    - high bias TIFF
    target = pf_ratio

    Expected CSV columns:
    - low_path
    - high_path
    - pf_ratio
    """

    def __init__(self, dataframe, image_size=224, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.image_size = image_size
        self.augment = augment

    def read_grayscale_tiff(self, path):
        """
        Read TIFF as grayscale using cv2.
        Even if stored as RGB, this forces single-channel reading.
        """
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            raise ValueError(f"Failed to load image: {path}")

        return img  # shape [H, W], dtype usually uint8

    def resize_image(self, img):
        """
        Resize image to fixed size using bilinear interpolation.
        """
        return cv2.resize(
            img,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_LINEAR
        )

    def apply_same_augmentation(self, low_img, high_img):
        """
        Apply the same random augmentation to both channels.
        Keep augmentation light for EL images.
        """
        # Horizontal flip
        if random.random() < 0.5:
            low_img = cv2.flip(low_img, 1)
            high_img = cv2.flip(high_img, 1)

        # Vertical flip
        if random.random() < 0.5:
            low_img = cv2.flip(low_img, 0)
            high_img = cv2.flip(high_img, 0)

        return low_img, high_img

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        low_path = row["low_path"]
        high_path = row["high_path"]
        target = row["pf_ratio"]

        # Read images
        low_img = self.read_grayscale_tiff(low_path)
        high_img = self.read_grayscale_tiff(high_path)

        # Resize
        low_img = self.resize_image(low_img)
        high_img = self.resize_image(high_img)

        # Same augmentation for both paired images
        if self.augment:
            low_img, high_img = self.apply_same_augmentation(low_img, high_img)

        # Normalize to [0, 1]
        low_img = low_img.astype(np.float32) / 255.0
        high_img = high_img.astype(np.float32) / 255.0

        # Add channel dimension: [1, H, W]
        low_img = np.expand_dims(low_img, axis=0)
        high_img = np.expand_dims(high_img, axis=0)

        # Stack into [2, H, W]
        x = np.concatenate([low_img, high_img], axis=0)

        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(target, dtype=torch.float32)

        return x, y


# =========================================
# MODEL
# =========================================
def build_efficientnet_b0_2ch(pretrained=True):
    """
    EfficientNet-B0 modified for:
    - 2 input channels
    - 1 regression output
    """
    if pretrained:
        weights = models.EfficientNet_B0_Weights.DEFAULT
    else:
        weights = None

    model = models.efficientnet_b0(weights=weights)

    # Original first conv layer
    old_conv = model.features[0][0]

    # New first conv layer for 2 channels
    new_conv = nn.Conv2d(
        in_channels=2,
        out_channels=old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=False
    )

    with torch.no_grad():
        if pretrained:
            # old_conv.weight shape = [out_channels, 3, k, k]
            old_weights = old_conv.weight

            # Simple initialization from first 2 RGB channels
            new_conv.weight[:, 0, :, :] = old_weights[:, 0, :, :]
            new_conv.weight[:, 1, :, :] = old_weights[:, 1, :, :]
        else:
            nn.init.kaiming_normal_(new_conv.weight, mode="fan_out", nonlinearity="relu")

    model.features[0][0] = new_conv

    # Replace classifier for regression
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, 1)

    return model


# =========================================
# TRAIN / EVALUATE FUNCTIONS
# =========================================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for inputs, targets in loader:
        inputs = inputs.to(device)                  # [B, 2, H, W]
        targets = targets.to(device).unsqueeze(1)  # [B] -> [B, 1]

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    preds = []
    trues = []

    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(device)
            targets = targets.to(device).unsqueeze(1)

            outputs = model(inputs)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * inputs.size(0)

            preds.extend(outputs.squeeze(1).cpu().numpy())
            trues.extend(targets.squeeze(1).cpu().numpy())

    loss_value = running_loss / len(loader.dataset)
    mae = mean_absolute_error(trues, preds)
    rmse = math.sqrt(mean_squared_error(trues, preds))
    r2 = r2_score(trues, preds)

    return loss_value, mae, rmse, r2, np.array(preds), np.array(trues)


# =========================================
# MAIN
# =========================================
def main():
    # -------------------------
    # Load CSV
    # -------------------------
    df = pd.read_csv(CSV_PATH)

    required_cols = ["low_path", "high_path", "pf_ratio"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column in CSV: {col}")

    # Optional path check
    for col in ["low_path", "high_path"]:
        missing = df[~df[col].apply(os.path.exists)]
        if len(missing) > 0:
            print(f"Warning: {len(missing)} missing files in column '{col}'")
            print(missing[[col]].head())

    # -------------------------
    # Train / validation split
    # -------------------------
    train_df, val_df = train_test_split(
        df,
        test_size=0.2,
        random_state=RANDOM_SEED
    )

    print(f"Train samples: {len(train_df)}")
    print(f"Val samples:   {len(val_df)}")

    # -------------------------
    # Dataset and DataLoader
    # -------------------------
    train_dataset = DualELDataset(train_df, image_size=IMAGE_SIZE, augment=True)
    val_dataset = DualELDataset(val_df, image_size=IMAGE_SIZE, augment=False)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS
    )

    # -------------------------
    # Build model
    # -------------------------
    model = build_efficientnet_b0_2ch(pretrained=True).to(DEVICE)

    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_rmse = float("inf")

    # -------------------------
    # Training loop
    # -------------------------
    for epoch in range(NUM_EPOCHS):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_mae, val_rmse, val_r2, _, _ = evaluate(model, val_loader, criterion, DEVICE)

        print(
            f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f} | "
            f"Val MAE: {val_mae:.6f} | "
            f"Val RMSE: {val_rmse:.6f} | "
            f"Val R2: {val_r2:.6f}"
        )

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_model_wts = copy.deepcopy(model.state_dict())

    # -------------------------
    # Load best model
    # -------------------------
    model.load_state_dict(best_model_wts)

    val_loss, val_mae, val_rmse, val_r2, preds, trues = evaluate(model, val_loader, criterion, DEVICE)

    print("\nBest Validation Performance")
    print(f"MAE  : {val_mae:.6f}")
    print(f"RMSE : {val_rmse:.6f}")
    print(f"R2   : {val_r2:.6f}")

    # -------------------------
    # Save model
    # -------------------------
    save_path = "efficientnet_b0_dual_el_pf_ratio_cv2.pth"
    torch.save(model.state_dict(), save_path)
    print(f"\nModel saved to: {save_path}")


if __name__ == "__main__":
    main()

In [ ]:
import os
import math
import copy
import random
import numpy as np
import pandas as pd
import cv2

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models

# Config 

In [ ]:
CSV_PATH = "pf_ratio_labels.csv"   # change to your CSV file
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
RANDOM_SEED = 42
NUM_WORKERS = 0   # start with 0 in Windows/Jupyter, can increase later

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Reproducibility

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)


# Dataset

In [ ]:
class DualELDataset(Dataset):
    """
    Dataset for paired EL images:
    - low bias TIFF
    - high bias TIFF
    target = pf_ratio

    Expected CSV columns:
    - low_path
    - high_path
    - pf_ratio
    """

    def __init__(self, dataframe, image_size=224, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.image_size = image_size
        self.augment = augment

    def read_grayscale_tiff(self, path):
        """
        Read TIFF as grayscale using cv2.
        Even if stored as RGB, this forces single-channel reading.
        """
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            raise ValueError(f"Failed to load image: {path}")

        return img  # shape [H, W], dtype usually uint8

    def resize_image(self, img):
        """
        Resize image to fixed size using bilinear interpolation.
        """
        return cv2.resize(
            img,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_LINEAR
        )

    def apply_same_augmentation(self, low_img, high_img):
        """
        Apply the same random augmentation to both channels.
        Keep augmentation light for EL images.
        """
        # Horizontal flip
        if random.random() < 0.5:
            low_img = cv2.flip(low_img, 1)
            high_img = cv2.flip(high_img, 1)

        # Vertical flip
        if random.random() < 0.5:
            low_img = cv2.flip(low_img, 0)
            high_img = cv2.flip(high_img, 0)

        return low_img, high_img

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        low_path = row["low_path"]
        high_path = row["high_path"]
        target = row["pf_ratio"]

        # Read images
        low_img = self.read_grayscale_tiff(low_path)
        high_img = self.read_grayscale_tiff(high_path)

        # Resize
        low_img = self.resize_image(low_img)
        high_img = self.resize_image(high_img)

        # Same augmentation for both paired images
        if self.augment:
            low_img, high_img = self.apply_same_augmentation(low_img, high_img)

        # Normalize to [0, 1]
        low_img = low_img.astype(np.float32) / 255.0
        high_img = high_img.astype(np.float32) / 255.0

        # Add channel dimension: [1, H, W]
        low_img = np.expand_dims(low_img, axis=0)
        high_img = np.expand_dims(high_img, axis=0)

        # Stack into [2, H, W]
        x = np.concatenate([low_img, high_img], axis=0)

        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(target, dtype=torch.float32)

        return x, y


# MODEL

In [ ]:
def build_efficientnet_b0_2ch(pretrained=True):
    """
    EfficientNet-B0 modified for:
    - 2 input channels
    - 1 regression output
    """
    if pretrained:
        weights = models.EfficientNet_B0_Weights.DEFAULT
    else:
        weights = None

    model = models.efficientnet_b0(weights=weights)

    # Original first conv layer
    old_conv = model.features[0][0]

    # New first conv layer for 2 channels
    new_conv = nn.Conv2d(
        in_channels=2,
        out_channels=old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=False
    )

    with torch.no_grad():
        if pretrained:
            # old_conv.weight shape = [out_channels, 3, k, k]
            old_weights = old_conv.weight

            # Simple initialization from first 2 RGB channels
            new_conv.weight[:, 0, :, :] = old_weights[:, 0, :, :]
            new_conv.weight[:, 1, :, :] = old_weights[:, 1, :, :]
        else:
            nn.init.kaiming_normal_(new_conv.weight, mode="fan_out", nonlinearity="relu")

    model.features[0][0] = new_conv

    # Replace classifier for regression
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, 1)

    return model


# TRAIN

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for inputs, targets in loader:
        inputs = inputs.to(device)                  # [B, 2, H, W]
        targets = targets.to(device).unsqueeze(1)  # [B] -> [B, 1]

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    preds = []
    trues = []

    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(device)
            targets = targets.to(device).unsqueeze(1)

            outputs = model(inputs)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * inputs.size(0)

            preds.extend(outputs.squeeze(1).cpu().numpy())
            trues.extend(targets.squeeze(1).cpu().numpy())

    loss_value = running_loss / len(loader.dataset)
    mae = mean_absolute_error(trues, preds)
    rmse = math.sqrt(mean_squared_error(trues, preds))
    r2 = r2_score(trues, preds)

    return loss_value, mae, rmse, r2, np.array(preds), np.array(trues)


# MAIN

In [ ]:
def main():
    # -------------------------
    # Load CSV
    # -------------------------
    df = pd.read_csv(CSV_PATH)

    required_cols = ["low_path", "high_path", "pf_ratio"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column in CSV: {col}")

    # Optional path check
    for col in ["low_path", "high_path"]:
        missing = df[~df[col].apply(os.path.exists)]
        if len(missing) > 0:
            print(f"Warning: {len(missing)} missing files in column '{col}'")
            print(missing[[col]].head())

    # -------------------------
    # Train / validation split
    # -------------------------
    train_df, val_df = train_test_split(
        df,
        test_size=0.2,
        random_state=RANDOM_SEED
    )

    print(f"Train samples: {len(train_df)}")
    print(f"Val samples:   {len(val_df)}")

    # -------------------------
    # Dataset and DataLoader
    # -------------------------
    train_dataset = DualELDataset(train_df, image_size=IMAGE_SIZE, augment=True)
    val_dataset = DualELDataset(val_df, image_size=IMAGE_SIZE, augment=False)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS
    )

    # -------------------------
    # Build model
    # -------------------------
    model = build_efficientnet_b0_2ch(pretrained=True).to(DEVICE)

    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_rmse = float("inf")

    # -------------------------
    # Training loop
    # -------------------------
    for epoch in range(NUM_EPOCHS):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_mae, val_rmse, val_r2, _, _ = evaluate(model, val_loader, criterion, DEVICE)

        print(
            f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f} | "
            f"Val MAE: {val_mae:.6f} | "
            f"Val RMSE: {val_rmse:.6f} | "
            f"Val R2: {val_r2:.6f}"
        )

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_model_wts = copy.deepcopy(model.state_dict())

    # -------------------------
    # Load best model
    # -------------------------
    model.load_state_dict(best_model_wts)

    val_loss, val_mae, val_rmse, val_r2, preds, trues = evaluate(model, val_loader, criterion, DEVICE)

    print("\nBest Validation Performance")
    print(f"MAE  : {val_mae:.6f}")
    print(f"RMSE : {val_rmse:.6f}")
    print(f"R2   : {val_r2:.6f}")

    # -------------------------
    # Save model
    # -------------------------
    save_path = "efficientnet_b0_dual_el_pf_ratio_cv2.pth"
    torch.save(model.state_dict(), save_path)
    print(f"\nModel saved to: {save_path}")


if __name__ == "__main__":
    main()

In [2]:
import os
import math
import copy
import random
import numpy as np
import pandas as pd
import cv2

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models


# =========================================================
# CONFIG
# =========================================================
CSV_PATH = "AnonDB.csv"          # original CSV file
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
RANDOM_SEED = 42
NUM_WORKERS = 0   # use 0 first, especially on Windows/Jupyter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =========================================================
# REPRODUCIBILITY
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)


# =========================================================
# HELPER FUNCTIONS
# =========================================================
def resolve_relative_path(path_str, base_dir, add_wrp=True):
    """
    Convert relative paths like './EL/3903_35_1_01272020_20.tiff'
    into absolute paths based on the CSV/script directory.

    If add_wrp=True, convert:
    3903_35_1_01272020_20.tiff
    -> 3903_35_1_01272020_20_wrp.tiff
    """
    if pd.isna(path_str):
        return None

    path_str = str(path_str).strip()

    if path_str == "":
        return None

    # Normalize slashes
    path_str = path_str.replace("\\", os.sep).replace("/", os.sep)

    # Add _wrp before file extension if needed
    if add_wrp:
        root, ext = os.path.splitext(path_str)
        if not root.endswith("_wrp"):
            path_str = root + "_wrp" + ext

    # If already absolute, keep it
    if os.path.isabs(path_str):
        return os.path.normpath(path_str)

    # Remove leading ./ if present
    if path_str.startswith("." + os.sep):
        path_str = path_str[2:]

    return os.path.normpath(os.path.join(base_dir, path_str))


def prepare_dataframe(csv_path):
    """
    Load original CSV, compute PF ratio, resolve paths, and
    keep only valid rows with existing low/high EL files.
    """
    base_dir = os.path.dirname(os.path.abspath(csv_path))
    df = pd.read_csv(csv_path)

    required_cols = ["ELLPath", "ELHPath", "Pmp_(W)", "Nameplate_Pmp_(W)"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column in CSV: {col}")

    # Convert power columns to numeric safely
    df["Pmp_(W)"] = pd.to_numeric(df["Pmp_(W)"], errors="coerce")
    df["Nameplate_Pmp_(W)"] = pd.to_numeric(df["Nameplate_Pmp_(W)"], errors="coerce")

    # Remove invalid denominator
    df = df[df["Nameplate_Pmp_(W)"].notna()]
    df = df[df["Nameplate_Pmp_(W)"] > 0]

    # Remove invalid numerator
    df = df[df["Pmp_(W)"].notna()]

    # Compute PF ratio
    df["pf_ratio"] = df["Pmp_(W)"] / df["Nameplate_Pmp_(W)"]

    # Resolve actual paths
    df["low_path"] = df["ELLPath"].apply(lambda p: resolve_relative_path(p, base_dir))
    df["high_path"] = df["ELHPath"].apply(lambda p: resolve_relative_path(p, base_dir))

    # Remove missing path strings
    df = df[df["low_path"].notna()]
    df = df[df["high_path"].notna()]

    # Keep only rows whose image files actually exist
    df["low_exists"] = df["low_path"].apply(os.path.exists)
    df["high_exists"] = df["high_path"].apply(os.path.exists)

    total_before = len(df)
    df = df[df["low_exists"] & df["high_exists"]].copy()
    total_after = len(df)

    print(f"Rows with valid power data before file filtering: {total_before}")
    print(f"Rows with both EL files existing: {total_after}")
    print(f"Rows removed due to missing image file(s): {total_before - total_after}")

    # Keep only needed columns
    keep_cols = ["low_path", "high_path", "pf_ratio"]
    optional_id_cols = [c for c in ["ID", "ModuleID", "Module", "Serial", "PanelID"] if c in df.columns]
    keep_cols = optional_id_cols + keep_cols

    return df[keep_cols].reset_index(drop=True)


# =========================================================
# DATASET
# =========================================================
class DualELDataset(Dataset):
    def __init__(self, dataframe, image_size=224, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.image_size = image_size
        self.augment = augment

    def read_grayscale_tiff(self, path):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            raise ValueError(f"Failed to load image: {path}")

        return img

    def resize_image(self, img):
        return cv2.resize(
            img,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_LINEAR
        )

    def apply_same_augmentation(self, low_img, high_img):
        """
        Keep augmentation identical for low/high pair.
        Keep it light for EL images.
        """
        if random.random() < 0.5:
            low_img = cv2.flip(low_img, 1)
            high_img = cv2.flip(high_img, 1)

        if random.random() < 0.5:
            low_img = cv2.flip(low_img, 0)
            high_img = cv2.flip(high_img, 0)

        return low_img, high_img

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        low_img = self.read_grayscale_tiff(row["low_path"])
        high_img = self.read_grayscale_tiff(row["high_path"])

        low_img = self.resize_image(low_img)
        high_img = self.resize_image(high_img)

        if self.augment:
            low_img, high_img = self.apply_same_augmentation(low_img, high_img)

        # Normalize to [0, 1]
        low_img = low_img.astype(np.float32) / 255.0
        high_img = high_img.astype(np.float32) / 255.0

        # [H, W] -> [1, H, W]
        low_img = np.expand_dims(low_img, axis=0)
        high_img = np.expand_dims(high_img, axis=0)

        # stack -> [2, H, W]
        x = np.concatenate([low_img, high_img], axis=0)

        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(row["pf_ratio"], dtype=torch.float32)

        return x, y


# =========================================================
# MODEL
# =========================================================
def build_efficientnet_b0_2ch(pretrained=True):
    """
    EfficientNet-B0 modified to accept 2-channel input
    and output 1 regression value.
    """
    if pretrained:
        weights = models.EfficientNet_B0_Weights.DEFAULT
    else:
        weights = None

    model = models.efficientnet_b0(weights=weights)

    old_conv = model.features[0][0]

    new_conv = nn.Conv2d(
        in_channels=2,
        out_channels=old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=False
    )

    with torch.no_grad():
        if pretrained:
            old_weights = old_conv.weight  # [out_channels, 3, k, k]

            # simple initialization using first 2 RGB channels
            new_conv.weight[:, 0, :, :] = old_weights[:, 0, :, :]
            new_conv.weight[:, 1, :, :] = old_weights[:, 1, :, :]
        else:
            nn.init.kaiming_normal_(new_conv.weight, mode="fan_out", nonlinearity="relu")

    model.features[0][0] = new_conv

    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, 1)

    return model


# =========================================================
# TRAIN / EVALUATE
# =========================================================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for inputs, targets in loader:
        inputs = inputs.to(device)
        targets = targets.to(device).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    return running_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    preds = []
    trues = []

    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(device)
            targets = targets.to(device).unsqueeze(1)

            outputs = model(inputs)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * inputs.size(0)

            preds.extend(outputs.squeeze(1).cpu().numpy())
            trues.extend(targets.squeeze(1).cpu().numpy())

    loss_value = running_loss / len(loader.dataset)
    mae = mean_absolute_error(trues, preds)
    rmse = math.sqrt(mean_squared_error(trues, preds))
    r2 = r2_score(trues, preds)

    return loss_value, mae, rmse, r2, np.array(preds), np.array(trues)


# =========================================================
# MAIN
# =========================================================
def main():
    # -------------------------
    # Prepare dataframe
    # -------------------------
    df = prepare_dataframe(CSV_PATH)

    if len(df) < 10:
        raise ValueError("Too few valid samples remain after filtering.")

    print("\nSample rows:")
    print(df.head())

    # -------------------------
    # Train / validation split
    # -------------------------
    train_df, val_df = train_test_split(
        df,
        test_size=0.2,
        random_state=RANDOM_SEED
    )

    print(f"\nTrain samples: {len(train_df)}")
    print(f"Val samples:   {len(val_df)}")

    # -------------------------
    # Dataset / loader
    # -------------------------
    train_dataset = DualELDataset(train_df, image_size=IMAGE_SIZE, augment=True)
    val_dataset = DualELDataset(val_df, image_size=IMAGE_SIZE, augment=False)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS
    )

    # -------------------------
    # Model
    # -------------------------
    model = build_efficientnet_b0_2ch(pretrained=True).to(DEVICE)

    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_rmse = float("inf")

    # -------------------------
    # Train loop
    # -------------------------
    for epoch in range(NUM_EPOCHS):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_mae, val_rmse, val_r2, _, _ = evaluate(model, val_loader, criterion, DEVICE)

        print(
            f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f} | "
            f"Val MAE: {val_mae:.6f} | "
            f"Val RMSE: {val_rmse:.6f} | "
            f"Val R2: {val_r2:.6f}"
        )

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_model_wts = copy.deepcopy(model.state_dict())

    # -------------------------
    # Final evaluation
    # -------------------------
    model.load_state_dict(best_model_wts)

    val_loss, val_mae, val_rmse, val_r2, preds, trues = evaluate(
        model, val_loader, criterion, DEVICE
    )

    print("\nBest Validation Performance")
    print(f"MAE  : {val_mae:.6f}")
    print(f"RMSE : {val_rmse:.6f}")
    print(f"R2   : {val_r2:.6f}")

    # -------------------------
    # Save model
    # -------------------------
    save_path = "efficientnet_b0_dual_el_pf_ratio_from_AnonDB.pth"
    torch.save(model.state_dict(), save_path)
    print(f"\nModel saved to: {save_path}")


if __name__ == "__main__":
    main()

0.6%

Rows with valid power data before file filtering: 582
Rows with both EL files existing: 576
Rows removed due to missing image file(s): 6

Sample rows:
                                            low_path  \
0  c:\Users\Ghozy Abror\OneDrive - Institut Tekno...   
1  c:\Users\Ghozy Abror\OneDrive - Institut Tekno...   
2  c:\Users\Ghozy Abror\OneDrive - Institut Tekno...   
3  c:\Users\Ghozy Abror\OneDrive - Institut Tekno...   
4  c:\Users\Ghozy Abror\OneDrive - Institut Tekno...   

                                           high_path  pf_ratio  
0  c:\Users\Ghozy Abror\OneDrive - Institut Tekno...  0.952963  
1  c:\Users\Ghozy Abror\OneDrive - Institut Tekno...  0.948889  
2  c:\Users\Ghozy Abror\OneDrive - Institut Tekno...  0.941111  
3  c:\Users\Ghozy Abror\OneDrive - Institut Tekno...  0.954074  
4  c:\Users\Ghozy Abror\OneDrive - Institut Tekno...  0.944074  

Train samples: 460
Val samples:   116
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810b

100.0%


Epoch [01/30] | Train Loss: 0.537243 | Val Loss: 0.412108 | Val MAE: 0.577604 | Val RMSE: 0.641956 | Val R2: -993.924139
Epoch [02/30] | Train Loss: 0.097922 | Val Loss: 0.078333 | Val MAE: 0.230943 | Val RMSE: 0.279880 | Val R2: -188.112934
Epoch [03/30] | Train Loss: 0.054424 | Val Loss: 0.073246 | Val MAE: 0.217245 | Val RMSE: 0.270639 | Val R2: -175.831631
Epoch [04/30] | Train Loss: 0.047756 | Val Loss: 0.055933 | Val MAE: 0.188515 | Val RMSE: 0.236501 | Val R2: -134.034916
Epoch [05/30] | Train Loss: 0.047066 | Val Loss: 0.041682 | Val MAE: 0.163808 | Val RMSE: 0.204161 | Val R2: -99.628862
Epoch [06/30] | Train Loss: 0.039149 | Val Loss: 0.037126 | Val MAE: 0.151040 | Val RMSE: 0.192681 | Val R2: -88.631132
Epoch [07/30] | Train Loss: 0.038139 | Val Loss: 0.030179 | Val MAE: 0.139380 | Val RMSE: 0.173721 | Val R2: -71.859249
Epoch [08/30] | Train Loss: 0.031798 | Val Loss: 0.031860 | Val MAE: 0.143145 | Val RMSE: 0.178495 | Val R2: -75.918321
Epoch [09/30] | Train Loss: 0.032373